In [ ]:
import pandas as pd
import datasets as DS
import numpy as np
from networkx.utils.union_find import UnionFind

FILEDIR: str = "data/STORE1/"

books = DS.loadDataset("books.yaml", FILEDIR)
orders = DS.loadDataset("orders.parquet", FILEDIR)
users = DS.loadDataset("users.csv", FILEDIR)

### Books

In [130]:
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         750 non-null    int64 
 1   title      750 non-null    str   
 2   author     750 non-null    str   
 3   genre      750 non-null    str   
 4   publisher  748 non-null    str   
 5   year       750 non-null    object
dtypes: int64(1), object(1), str(4)
memory usage: 83.4+ KB


In [131]:
books.head()

,id,title,author,genre,publisher,year
0,19437,The Grand Budapest Hotel,"Myles Rohan Sr., Shirl Conn, Rep. Charley Hickle",Suspense/Thriller,Carlton Publishing Group,1980
1,19267,All the King's Men,"Jackie Ondricka, Jewell Harvey JD",Classic,Harvill Press at Random House,1995
2,19428,Once Upon a Time in the West,Sana Marks,Crime/Detective,Legend Books,2004
3,19183,A Swiftly Tilting Planet,Normand Gusikowski,Folklore,,1928
4,19491,WarioWare: Touched!,"Patsy Rice, Leola Mills",Mythopoeia,Eel Pie Publishing,2024


In [132]:
books.tail()

,id,title,author,genre,publisher,year
745,18998,Dulce et Decorum Est,Edwin Herzog JD,Historical fiction,Hogarth Press,1837
746,18943,Super Mario World,Virgen Goldner,Mythopoeia,Hachette Book Group USA,2018
747,19196,Terrible Swift Sword,Etha Schiller,Fiction narrative,Jarrolds Publishing,2007
748,19132,Star Wars: Episode VI - Return of the Jedi,"Shantell Friesen PhD, Rep. Magda Kemmer, Thers...",Crime/Detective,Bowes & Bowes,1986
749,18871,The Soldier's Art,"Valentin Kutch, Nery Romaguera",Mythology,Chambers Harrap,2002


In [133]:
books.drop_duplicates(inplace = True)
books.shape

(750, 6)

In [134]:
books["year"].unique()

array([1980, 1995, 2004, 1928, 2024, 1831, 2000, 2013, 1985, 2011, 2019,
       1982, 2003, 1986, 1983, 2002, 2016, 1993, 1979, 1992, 1988, 1994,
       1976, 1998, 1955, 2008, 1967, 1978, 1987, 2026, 1977, 1984, 1989,
       2012, 2005, 2022, 1990, 2020, 2015, 2010, 1991, 1959, '-1', 2009,
       2014, 2025, 2018, 2023, 1970, 2007, 1834, 1971, 2001, 1981, 1852,
       1860, 1957, 1997, 1934, 1996, 1901, 2006, 1877, 2017, 1961, 1950,
       1964, 1965, 1899, 1968, 2021, 0, 1962, 1895, 1999, 1924, 1892,
       1974, 1840, 1973, 1830, 1919, 1846, 1927, 1937, 1943, 1951, 1863,
       1972, 1839, 1975, -1, 1936, ' ', 1881, 1896, 1886, 1969, 1966,
       1931, '-', '', 1948, 1837], dtype=object)

In [135]:
year_str = books["year"].astype(str).str.strip()
idx_correct_years = year_str.str.match(r'^\d{4}$', na=False) # regex for finding only 4 digit number
not_years = books[~idx_correct_years]["year"].unique()       # ~ negates (inverts)
books["year"] = books["year"].replace(not_years, np.nan).astype("Int16")

In [136]:
books[books["publisher"].str.len() < 8]["publisher"].unique()

<ArrowStringArray>
[     ' ',  'Brill', 'Newnes',       '', 'FabJob',     '\t',  'Tammi',
  'Nauka',      '-',   'HMSO',   'NULL']
Length: 11, dtype: str

In [137]:
for bad_value in ['-', '', ' ', '\t', 'NULL']:
    books["publisher"] = books["publisher"].replace(bad_value, pd.NA)

books[books["publisher"].str.len() < 8]["publisher"].unique()

<ArrowStringArray>
['Brill', 'Newnes', 'FabJob', 'Tammi', 'Nauka', 'HMSO']
Length: 6, dtype: str

In [138]:
books["publisher"].isna().sum()

np.int64(11)

In [139]:
books[books["publisher"].isna()]

,id,title,author,genre,publisher,year
3,19183,A Swiftly Tilting Planet,Normand Gusikowski,Folklore,NaN,1928
126,19182,Monty Python and the Holy Grail,"Dorian Boyer, Zane Bartell",Textbook,NaN,1998
226,18925,The Pianist,Barney Will,Fiction in verse,NaN,1982
253,19454,Everything is Illuminated,Maryjo Howell,Suspense/Thriller,NaN,2024
254,18860,The Wizard of Oz,Daria Schiller JD,Fantasy,NaN,1984
431,19477,Dishonored,Ryann Erdman,Historical fiction,NaN,1998
436,18870,Wildfire at Midnight,Genny Toy,Mythology,NaN,1977
486,19104,Backstreet Boys,"Harold Huels, Maya Grant",Mythopoeia,NaN,1981
585,19431,Crash Bandicoot,Leslee Kshlerin,Biography/Autobiography,NaN,1979
628,19045,It's a Battlefield,Gertrudis Stanton,Fantasy,NaN,1989


In [140]:
def sortAuthors(authors:str) -> str:
    return ", ".join(sorted([author.strip() for author in authors.split(",")]))
books["author"] = books["author"].apply(sortAuthors)
len(books["author"].unique())

311

### Users

In [141]:
users.drop_duplicates(inplace = True)
users.info()
users.shape

<class 'pandas.DataFrame'>
RangeIndex: 3158 entries, 0 to 3157
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       3158 non-null   int64
 1   name     3158 non-null   str  
 2   address  3046 non-null   str  
 3   phone    3158 non-null   str  
 4   email    3158 non-null   str  
dtypes: int64(1), str(4)
memory usage: 433.3 KB


(3158, 5)

In [142]:
def showDuplicated(data: pd.DataFrame):
    for col in data.columns:
        print(f"Duplicated in {col}:", data[col].dropna().duplicated().sum())

# print("Duplicated users id:", users["id"].duplicated().sum())
# print("Duplicated users name:", users["name"].duplicated().sum())
# print("Duplicated users address:", users["address"].duplicated().sum())
# print("Duplicated users phone:", users["phone"].duplicated().sum())
# print("Duplicated users email:", users["email"].duplicated().sum())

showDuplicated(users)

Duplicated in id: 0
Duplicated in name: 125
Duplicated in address: 169
Duplicated in phone: 126
Duplicated in email: 132


In [143]:
def showMissing(data: pd.DataFrame):
    for col in data.columns:
        print(f"Missing in {col}:", data[col].isna().sum())

# print("Missing users id:", users["id"].isna().sum())
# print("Missing users name:", users["name"].isna().sum())
# print("Missing users address:", users["address"].isna().sum())
# print("Missing users phone:", users["phone"].isna().sum())
# print("Missing users email:", users["email"].isna().sum())

showMissing(users)

Missing in id: 0
Missing in name: 0
Missing in address: 112
Missing in phone: 0
Missing in email: 0


In [144]:
users["address"] = users["address"].str.strip().replace('', None)
print("Missing users address:", users["address"].isna().sum())

Missing users address: 152


In [145]:
users["email"].unique()

<ArrowStringArray>
[              'guy.jacobson@schumm.example',
        'marshall_johnston@homenick.example',
                         'kurtis@stehr.test',
             'joannie.larson@gulgowski.test',
                    'earl_moen@flatley.test',
             'jerrold.murphy@kertzmann.test',
             'lauran@marvin-keebler.example',
        'colby_cartwright@prohaska-rau.test',
                        'arron@fritsch.test',
                  'archie@aufderhar.example',
 ...
                      'karol@hammes.example',
 'loren.langworth@thompson-oberbrunner.test',
                'bettina.stanton@huels.test',
              'blaine@abshire-schaefer.test',
          'cheyenne@goldner-collier.example',
                 'ariel@willms-osinski.test',
            'stephnie@wunsch-abernathy.test',
             'edythe_mccullough@russel.test',
                   'myrta@wilkinson.example',
                  'collen_kuhlman@will.test']
Length: 3026, dtype: str

In [146]:
users["phone"] = users["phone"].str.strip().str.replace(r'\D', '', regex = True)
users["phone"].head()

0    9887240521
1    9391482312
2    7249432841
3    3363840025
4    8165223840
Name: phone, dtype: str

#### User aliases
Some rows in `users` represent the same person. We assume that one of the fields: `[name, address, phone, email]` may change.

In [147]:
def uniqueUsers(users:pd.DataFrame):
    users = users.sort_values(by="id").reset_index(drop=True)
    uf = UnionFind(users["id"])
    users_dict = users.to_dict("records")       # main speed up
    
    columns = ["name", "address", "phone", "email"]

    def matches(user1, user2) -> int:
        return sum([user1[col] == user2[col] for col in columns])
        
    for i in range(len(users_dict)):
        for j in range(i + 1, len(users_dict)):
            if matches(users_dict[i], users_dict[j]) >= 3:
                uf.union(users_dict[i]["id"], users_dict[j]["id"])
    return list(uf.to_sets())

all_unique_users = uniqueUsers(users)

In [148]:
master_ids = dict()
for unique_users_set in all_unique_users:
    master_id = min(unique_users_set)
    for user_id in unique_users_set:
        master_ids[user_id] = master_id

users["master_id"] = users["id"].apply(lambda u: master_ids[u])
users.head()

,id,name,address,phone,email,master_id
0,53162,Arnoldo Moore,"Suite 617 27446 Tremblay Divide, Gislasonside,...",9887240521,guy.jacobson@schumm.example,53162
1,52237,Kenisha Zemlak Sr.,"2634 Fahey Forks, East Vance, CO 54477",9391482312,marshall_johnston@homenick.example,52237
2,53589,Floy Kling,NaN,7249432841,kurtis@stehr.test,53589
3,54329,Desmond Adams,"Apt. 594 4933 Walsh Vista, West Brandenside, L...",3363840025,joannie.larson@gulgowski.test,54329
4,53290,Torrie Sipes,"Apt. 442 4822 Sid Plaza, Dietrichhaven, TN 67794",8165223840,earl_moen@flatley.test,53290


### Orders

In [149]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 9714 entries, 0 to 9713
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          9714 non-null   int64
 1   user_id     9714 non-null   int64
 2   book_id     9714 non-null   int64
 3   quantity    9714 non-null   int32
 4   unit_price  9714 non-null   str  
 5   timestamp   9714 non-null   str  
 6   shipping    7349 non-null   str  
dtypes: int32(1), int64(3), str(3)
memory usage: 875.6 KB


In [150]:
showDuplicated(orders)
showMissing(orders)

Duplicated in id: 0
Duplicated in user_id: 6715
Duplicated in book_id: 8964
Duplicated in quantity: 9709
Duplicated in unit_price: 3940
Duplicated in timestamp: 0
Duplicated in shipping: 5755
Missing in id: 0
Missing in user_id: 0
Missing in book_id: 0
Missing in quantity: 0
Missing in unit_price: 0
Missing in timestamp: 0
Missing in shipping: 2365


In [151]:
def isEuro(price:str) -> bool:
    if "€" in price:
        return True
    
    price = price.strip().lower()
    
    if "eur" in price:
        return True
    
    return False

def isDollar(price:str) -> bool:
    if "$" in price:
        return True
    
    price = price.strip().lower()
    
    if "usd" in price:
        return True
    
    return False

len(orders[~orders["unit_price"].apply(lambda p: isEuro(p) or isDollar(p))]) # see if after division no other price formats are left

0

In [152]:
orders[orders["unit_price"].apply(lambda p: isDollar(p))]

,id,user_id,book_id,quantity,unit_price,timestamp,shipping
0,70365,51860,19516,1,$55¢50,"20:13,31-May-2026",NULL
1,75931,52316,19014,2,$ 50.75,01:08:20 P.M.;2025-12-28,
2,75546,54009,19457,1,$23.75,08:00:36 AM;2025-11-13,
4,74204,51627,19108,1,48.50USD,"09:46,26-Jul-2025","100 Balistreri Camp, Auerfurt, MO 30489"
5,71334,52620,19131,1,USD 22.5,"2026-06-22,13:21:39",
...,...,...,...,...,...,...,...
9703,77243,52628,18930,1,USD48.50,2026-01-14;17:12:30,
9704,71791,52860,19051,1,39USD,"02/11/26, 08:00:12 AM","13875 Donnelly Spring, Krajciktown, OR 27112"
9711,72466,54180,19249,2,45.5$,03/26/26 11:24:02 am,NaN
9712,78562,52049,19269,1,38.50 $,2026-01-25T01:11:04.956,"8843 Gulgowski Divide, Lake Julesside, NV 89152"


In [153]:
orders["quantity"].value_counts() # all good there

quantity
1    6796
2    1658
3     647
5     334
4     279
Name: count, dtype: int64

In [154]:
def toDollar(price:str) -> float:
    RATE = 1.2
    def handleCents(price:str) -> str:
        if "¢" in price:
            if price[-1] == "¢":
                price = f"{price[0:2]}.{price[2:-1]}"
            else: price = price.replace("¢", ".")
        return price
    
    if isEuro(price):
        price = price.lower().replace("€", "").replace("eur", "").strip()
        price = handleCents(price)
        usd_price: float = round(RATE * float(price), 2)
    else:
        price = price.lower().replace("$", "").replace("usd", "").strip()
        price = handleCents(price)
        usd_price: float = round(float(price), 2)
    return usd_price

orders["paid_price"] = orders["unit_price"].apply(toDollar) * orders["quantity"]
orders.head(7)

,id,user_id,book_id,quantity,unit_price,timestamp,shipping,paid_price
0,70365,51860,19516,1,$55¢50,"20:13,31-May-2026",NULL,55.50
1,75931,52316,19014,2,$ 50.75,01:08:20 P.M.;2025-12-28,,101.50
2,75546,54009,19457,1,$23.75,08:00:36 AM;2025-11-13,,23.75
3,74431,53245,19161,1,30.25 €,19:39:13 14-August-2025,"Suite 587 7406 Ahmed Meadows, Spencershire, NM...",36.30
4,74204,51627,19108,1,48.50USD,"09:46,26-Jul-2025","100 Balistreri Camp, Auerfurt, MO 30489",48.50
5,71334,52620,19131,1,USD 22.5,"2026-06-22,13:21:39",,22.50
6,71634,51279,19408,3,63.$,2025-12-11T02:01:49,NULL,189.00


In [155]:
weird_shippings = orders[orders["shipping"].str.len() < 15]["shipping"].unique().tolist()
for val in weird_shippings:
    orders.replace({"shipping": val}, pd.NA, inplace=True)
weird_shippings

['NULL', '', ' ']

In [156]:
orders["timestamp"].head(10)

0           20:13,31-May-2026
1    01:08:20 P.M.;2025-12-28
2      08:00:36 AM;2025-11-13
3     19:39:13 14-August-2025
4           09:46,26-Jul-2025
5         2026-06-22,13:21:39
6         2025-12-11T02:01:49
7         2026-06-17T02:46:46
8    21-OCT-2025, 09:47:23 AM
9            19:59 2025-07-22
Name: timestamp, dtype: str

In [157]:
from dateutil import parser

def parseDate(date:str) -> str | pd.api.typing.NaTType:
    date = date.replace(",", " ")                                               # main culprit - ","
    try:
        return str(parser.parse(date, ignoretz=True, fuzzy=True)).split(".")[0] # throwing out miliseconds
    except (ValueError, TypeError, OverflowError):
        return pd.NaT

orders["date"] = pd.to_datetime(orders["timestamp"].map(parseDate), errors="coerce").dt.strftime("%Y-%m-%d")
orders["date"].isna().sum()

np.int64(0)

In [158]:
orders["date"] = orders["date"].astype("datetime64[s]")

In [159]:
orders.head()

,id,user_id,book_id,quantity,unit_price,timestamp,shipping,paid_price,date
0,70365,51860,19516,1,$55¢50,"20:13,31-May-2026",NaN,55.50,2026-05-31
1,75931,52316,19014,2,$ 50.75,01:08:20 P.M.;2025-12-28,NaN,101.50,2025-12-28
2,75546,54009,19457,1,$23.75,08:00:36 AM;2025-11-13,NaN,23.75,2025-11-13
3,74431,53245,19161,1,30.25 €,19:39:13 14-August-2025,"Suite 587 7406 Ahmed Meadows, Spencershire, NM...",36.30,2025-08-14
4,74204,51627,19108,1,48.50USD,"09:46,26-Jul-2025","100 Balistreri Camp, Auerfurt, MO 30489",48.50,2025-07-26


In [160]:
books.to_csv("./cleaned_" + FILEDIR + "books.csv", index=False)
users.to_csv("./cleaned_" + FILEDIR + "users.csv", index=False)
orders.to_csv("./cleaned_" + FILEDIR + "orders.csv", index=False)